In [0]:
# ============================================================
# 03_gold_aggregate_0604 — Gold aggregation via MERGE
# Author: oakville3456
# Updated: 2026-06-06
# Branch: main
# Purpose: Aggregate Silver → Gold (daily revenue by store)
#          using Delta MERGE via Unity Catalog
# ============================================================

from pyspark.sql import functions as F

# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
SILVER_TBL = "adb_retail_dev.silver.sales"
GOLD_TBL   = "adb_retail_dev.gold.sales_daily"

# ─────────────────────────────────────────────────────────────
# READ SILVER
# ─────────────────────────────────────────────────────────────
silver = spark.read.table(SILVER_TBL)

print(f"Silver rows read: {silver.count()}")

# ─────────────────────────────────────────────────────────────
# AGGREGATE DAILY METRICS
# countDistinct is fine here — this is a batch read, not streaming
# (streaming requires approx_count_distinct)
# ─────────────────────────────────────────────────────────────
gold_updates = (
    silver
    .filter(F.col("order_date").isNotNull())        # exclude rows with bad dates
    .groupBy("store_id", "order_date")
    .agg(
        F.round(F.sum("revenue"), 2).alias("total_revenue"),
        F.count("order_id").alias("order_count"),
        F.countDistinct("customer_id").alias("unique_customers"),
        F.round(F.avg("revenue"), 2).alias("avg_order_value")
    )
)

print(f"Gold update rows (store+date combinations): {gold_updates.count()}")

# ─────────────────────────────────────────────────────────────
# CREATE GOLD TABLE IF NOT EXISTS
# Safe to run on first run or any subsequent run
# ─────────────────────────────────────────────────────────────
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {GOLD_TBL} (
        store_id          STRING,
        order_date        DATE,
        total_revenue     DOUBLE,
        order_count       BIGINT,
        unique_customers  BIGINT,
        avg_order_value   DOUBLE
    )
    USING DELTA
    COMMENT 'Daily revenue aggregated by store and date.
     Source: Silver sales table.
     MERGE key: store_id + order_date.
     Updated via Delta MERGE upsert pattern.'
""")

# ─────────────────────────────────────────────────────────────
# MERGE INTO GOLD (incremental)
# - match on composite key: store_id + order_date
# - existing rows → UPDATE all metrics
# - new rows      → INSERT
# - idempotent: running twice produces same result
# ─────────────────────────────────────────────────────────────
(
    gold_updates.alias("u")
    .merge(
        spark.table(GOLD_TBL).alias("g"),
        "u.store_id = g.store_id AND u.order_date = g.order_date"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

# ─────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────
gold = spark.table(GOLD_TBL)
gold_count = gold.count()

print("────────────────────────────────────────────")
print(f" Gold rows: {gold_count}  (stores × dates)")
print("────────────────────────────────────────────")

display(gold.orderBy("order_date", "store_id"))